# LED controlado por Teachable Machine

Este notebook carrega um modelo de imagem treinado no [Teachable Machine](https://teachablemachine.withgoogle.com/)
no formato **Keras (`.h5`)** e acende ou apaga um **LED virtual** conforme a classe reconhecida pela webcam.

**Ordem de execucao:**

1. `Ambiente` - confere as versoes.
2. `Enviar o modelo` - faz upload do `converted_keras.zip` (ou do `keras_model.h5` + `labels.txt`).
3. `Carregar o modelo` - le o `.h5` e as classes.
4. `Configuracao` - escolhe qual classe liga o LED.
5. `Funcoes` - pre-processamento e previsao.
6. `Demo por foto` **ou** `Demo ao vivo`.

> Exporte no Teachable Machine em **Export Model -> Tensorflow -> Keras -> Download my model**.

> Notebook da apresentacao:
> <https://colab.research.google.com/drive/1F_z4W5o1wA8a3vgxpawteX4MQVAq9Sh7>

## 1. Ambiente

In [ ]:
import sys, tensorflow as tf, numpy as np, PIL

print("Python     :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("Keras      :", tf.keras.__version__)
print("NumPy      :", np.__version__)
print("Pillow     :", PIL.__version__)

## 2. Enviar o modelo

Se o modelo estiver publicado no GitHub, esta celula **baixa tudo sozinha** e nao precisa fazer nada.
Se o download falhar (ou se `URL_MODELO` estiver vazia), ela abre o seletor de arquivos para
voce enviar o `converted_keras.zip` que o Teachable Machine baixou.

In [ ]:
import os, re, glob, zipfile, urllib.request

# De onde baixar o converted_keras.zip. Aceita link do GitHub (raw) ou do Google Drive
# compartilhado como "qualquer pessoa com o link". Deixe "" para sempre enviar na mao.
URL_MODELO = "https://raw.githubusercontent.com/zmixtv1/Estudos/main/TeachableMachineLED/converted_keras.zip"

PACOTE = "converted_keras.zip"


def url_direta(url):
    # Converte link de compartilhamento do Google Drive em link de download direto.
    if "drive.google.com" not in url:
        return url
    achado = (re.search("/file/d/([A-Za-z0-9_-]+)", url)
              or re.search("[?&]id=([A-Za-z0-9_-]+)", url))
    return f"https://drive.google.com/uc?export=download&id={achado.group(1)}" if achado else url


def baixar_modelo(url, destino=PACOTE):
    urllib.request.urlretrieve(url_direta(url), destino)
    if not zipfile.is_zipfile(destino):        # 404 ou pagina HTML no lugar do zip
        os.remove(destino)
        raise ValueError("o arquivo baixado nao e um zip valido")
    return destino


pacotes = []

if URL_MODELO:
    try:
        pacotes.append(baixar_modelo(URL_MODELO))
        print("Modelo baixado do GitHub.")
    except Exception as e:
        print(f"Download falhou ({type(e).__name__}: {e}). Envie o arquivo manualmente.")

if not pacotes and not os.path.exists("keras_model.h5"):
    try:
        from google.colab import files
        pacotes += list(files.upload())         # abre o seletor de arquivos
    except ImportError:
        print("Nao esta no Colab - usando os arquivos da pasta atual.")

# Extrai o zip do Teachable Machine (contem keras_model.h5 e labels.txt).
for nome in pacotes or glob.glob("*.zip"):
    if nome.lower().endswith(".zip"):
        with zipfile.ZipFile(nome) as z:
            z.extractall()
        print("Extraido:", nome, "->", z.namelist())

print()
print("Arquivos disponiveis:")
for f in sorted(os.listdir()):
    if f.endswith((".h5", ".txt", ".zip")):
        print(" -", f, f"({os.path.getsize(f)/1024:.0f} KB)")

## 3. Carregar o modelo

O `.h5` do Teachable Machine foi salvo com uma versao antiga do Keras, e o Colab hoje vem com o Keras 3.
Por isso a funcao abaixo tenta tres estrategias, da mais simples para a mais tolerante, e avisa qual funcionou:

1. carregamento normal;
2. carregamento com a camada `DepthwiseConv2D` corrigida (ignora o argumento `groups`, que o Keras 3 nao aceita);
3. carregamento pelo pacote `tf_keras`, que e o Keras 2 mantido pelo Google para arquivos legados.

In [ ]:
import sys
import tensorflow as tf

CAMINHO_MODELO = "keras_model.h5"
CAMINHO_LABELS = "labels.txt"


def _depthwise_compativel(base):
    """DepthwiseConv2D que ignora o argumento 'groups' presente nos modelos antigos."""
    class DepthwiseConv2DCompativel(base):
        def __init__(self, *args, **kwargs):
            kwargs.pop("groups", None)
            super().__init__(*args, **kwargs)
    return DepthwiseConv2DCompativel


def carregar_modelo(caminho=CAMINHO_MODELO):
    erros = []

    # 1) caminho feliz
    try:
        modelo = tf.keras.models.load_model(caminho, compile=False)
        print("Modelo carregado normalmente.")
        return modelo
    except Exception as e:
        erros.append(("padrao", e))

    # 2) mesma coisa, corrigindo a DepthwiseConv2D
    try:
        base = tf.keras.layers.DepthwiseConv2D
        modelo = tf.keras.models.load_model(
            caminho, compile=False,
            custom_objects={"DepthwiseConv2D": _depthwise_compativel(base)},
        )
        print("Modelo carregado com a correcao da camada DepthwiseConv2D.")
        return modelo
    except Exception as e:
        erros.append(("DepthwiseConv2D corrigida", e))

    # 3) ultimo recurso: Keras 2 via tf_keras
    try:
        try:
            import tf_keras
        except ImportError:
            import subprocess
            print("Instalando tf_keras (Keras 2)...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf_keras"], check=True)
            import tf_keras
        try:
            modelo = tf_keras.models.load_model(caminho, compile=False)
        except Exception:
            base = tf_keras.layers.DepthwiseConv2D
            modelo = tf_keras.models.load_model(
                caminho, compile=False,
                custom_objects={"DepthwiseConv2D": _depthwise_compativel(base)},
            )
        print("Modelo carregado com tf_keras (Keras 2).")
        return modelo
    except Exception as e:
        erros.append(("tf_keras", e))

    print("Nenhuma estrategia funcionou:\n")
    for nome, e in erros:
        print(f"[{nome}] {type(e).__name__}: {e}\n")
    raise RuntimeError("Nao foi possivel carregar o modelo - veja os erros acima.")


modelo = carregar_modelo()

# labels.txt vem no formato "0 Nome da classe" - tiramos o numero da frente.
with open(CAMINHO_LABELS, encoding="utf-8") as f:
    CLASSES = [linha.strip().split(" ", 1)[-1] for linha in f if linha.strip()]

# Tamanho de entrada esperado pelo modelo (o Teachable Machine usa 224x224).
try:
    ALTURA, LARGURA = modelo.input_shape[1], modelo.input_shape[2]
except Exception:
    ALTURA = LARGURA = 224

print("\nEntrada do modelo:", (ALTURA, LARGURA))
print("Classes encontradas:")
for i, nome in enumerate(CLASSES):
    print(f"  [{i}] {nome}")

## 4. Configuracao

Ajuste aqui e rode de novo se quiser mudar o comportamento na hora da apresentacao.

In [ ]:
# Qual classe acende o LED: use o indice (ex.: 0) ou o nome exato (ex.: "Mao aberta").
CLASSE_LIGA = 0

# Confianca minima para valer a deteccao (0 a 1).
LIMIAR = 0.70

# "hold"   -> LED aceso somente enquanto a classe esta sendo detectada
# "toggle" -> cada nova deteccao inverte o estado do LED
MODO = "hold"


def indice_da_classe(alvo):
    if isinstance(alvo, int):
        return alvo
    nomes = [c.lower() for c in CLASSES]
    return nomes.index(str(alvo).lower())


IDX_LIGA = indice_da_classe(CLASSE_LIGA)
print(f"LED liga com a classe [{IDX_LIGA}] {CLASSES[IDX_LIGA]!r} "
      f"| limiar {LIMIAR:.0%} | modo {MODO}")

## 5. Funcoes de previsao e do LED

O pre-processamento e o mesmo que o Teachable Machine usa: recorta o centro da imagem,
redimensiona para 224x224 e normaliza os pixels para o intervalo -1 a 1.

In [ ]:
import numpy as np
from PIL import Image, ImageOps

REAMOSTRAGEM = getattr(Image, "Resampling", Image).LANCZOS


def preprocessar(imagem):
    """PIL.Image -> array (1, ALTURA, LARGURA, 3) normalizado em [-1, 1]."""
    imagem = ImageOps.fit(imagem.convert("RGB"), (LARGURA, ALTURA), REAMOSTRAGEM)
    arr = np.asarray(imagem, dtype=np.float32)
    arr = (arr / 127.5) - 1.0
    return arr.reshape(1, ALTURA, LARGURA, 3)


def prever(imagem):
    """Retorna o vetor de probabilidades de cada classe."""
    return modelo.predict(preprocessar(imagem), verbose=0)[0]


# Estado do LED entre um quadro e outro (usado pelo modo "toggle").
led_ligado = False
estava_acima = False


def decidir_led(probs):
    """Aplica limiar e modo, atualiza o estado e devolve (ligado, indice_vencedor)."""
    global led_ligado, estava_acima

    acima = bool(probs[IDX_LIGA] >= LIMIAR)
    if MODO == "hold":
        led_ligado = acima
    elif acima and not estava_acima:      # so inverte na borda de subida
        led_ligado = not led_ligado
    estava_acima = acima

    return led_ligado, int(np.argmax(probs))


def resetar_led():
    global led_ligado, estava_acima
    led_ligado = estava_acima = False

## 6. Demo por foto

O jeito mais seguro de apresentar: tira uma foto pela webcam, classifica e mostra o LED.
Rode a celula quantas vezes quiser.

In [ ]:
import base64, io
from IPython.display import display, HTML, Javascript

try:
    from google.colab.output import eval_js
except ImportError:
    eval_js = None


def tirar_foto(qualidade=0.85):
    """Mostra a webcam com um botao 'Capturar' e devolve a foto como PIL.Image."""
    display(Javascript('''
      async function tirarFoto(qualidade) {
        const div = document.createElement('div');
        const video = document.createElement('video');
        const botao = document.createElement('button');
        botao.textContent = 'Capturar';
        botao.style.cssText = 'display:block;margin:8px 0;padding:8px 16px;font-size:14px;cursor:pointer';

        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        div.appendChild(botao);
        video.srcObject = stream;
        video.style.cssText = 'max-width:320px;border-radius:8px';
        await video.play();

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => botao.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getTracks().forEach(t => t.stop());
        div.remove();
        return canvas.toDataURL('image/jpeg', qualidade);
      }
    '''))
    dados = eval_js(f"tirarFoto({qualidade})")
    binario = base64.b64decode(dados.split(",", 1)[1])
    return Image.open(io.BytesIO(binario))


def mostrar_led(ligado, probs):
    """Desenha o LED e as barras de confianca na saida da celula."""
    cor = "#ff4136" if ligado else "#3a1010"
    brilho = "0 0 28px 8px rgba(255,65,54,.65)" if ligado else "none"
    texto = "LIGADO" if ligado else "DESLIGADO"

    barras = ""
    for i, (nome, p) in enumerate(zip(CLASSES, probs)):
        destaque = "#4c8dff" if i == IDX_LIGA else "#888"
        barras += (
            f"<div style='margin:6px 0;font-family:sans-serif;font-size:13px'>"
            f"<div style='display:flex;justify-content:space-between;max-width:280px'>"
            f"<span>{nome}</span><span>{p:.0%}</span></div>"
            f"<div style='max-width:280px;height:8px;background:#eee;border-radius:4px;overflow:hidden'>"
            f"<div style='width:{p*100:.1f}%;height:100%;background:{destaque}'></div></div></div>"
        )

    display(HTML(
        f"<div style='display:flex;gap:24px;align-items:center'>"
        f"<div style='text-align:center;font-family:sans-serif'>"
        f"<div style='width:110px;height:110px;border-radius:50%;background:{cor};"
        f"border:6px solid #23272f;box-shadow:{brilho};margin:0 auto 10px'></div>"
        f"<div style='font-weight:700;letter-spacing:2px'>{texto}</div></div>"
        f"<div>{barras}</div></div>"
    ))


foto = tirar_foto()
probs = prever(foto)
ligado, vencedora = decidir_led(probs)

print(f"Classe reconhecida: {CLASSES[vencedora]} ({probs[vencedora]:.1%})")
display(foto.resize((240, 180)))
mostrar_led(ligado, probs)

## 7. Demo ao vivo (opcional)

Aqui a webcam fica ligada e o LED acende sozinho, sem precisar clicar em nada.
Rode a celula e faca o gesto na frente da camera. Para encerrar antes do tempo,
use o botao de parar (o quadrado) do proprio Colab.

In [ ]:
import json, time, base64, io
from PIL import Image
from IPython.display import display, Javascript

def demo_ao_vivo(segundos=60):
    display(Javascript('''
      var _video, _stream, _canvas, _led, _txt, _info;

      async function iniciarVideo() {
        const div = document.createElement('div');
        div.style.cssText = 'display:flex;gap:24px;align-items:center;font-family:sans-serif';
        document.body.appendChild(div);

        _video = document.createElement('video');
        _video.style.cssText = 'max-width:320px;border-radius:8px';
        _stream = await navigator.mediaDevices.getUserMedia({video: true});
        _video.srcObject = _stream;
        await _video.play();
        div.appendChild(_video);

        const painel = document.createElement('div');
        painel.style.cssText = 'text-align:center';
        _led = document.createElement('div');
        _led.style.cssText = 'width:110px;height:110px;border-radius:50%;background:#3a1010;' +
                             'border:6px solid #23272f;margin:0 auto 10px;transition:all .15s';
        _txt = document.createElement('div');
        _txt.textContent = 'DESLIGADO';
        _txt.style.cssText = 'font-weight:700;letter-spacing:2px';
        _info = document.createElement('div');
        _info.style.cssText = 'font-size:13px;color:#666;margin-top:6px';
        painel.appendChild(_led); painel.appendChild(_txt); painel.appendChild(_info);
        div.appendChild(painel);

        _canvas = document.createElement('canvas');
        _canvas.width = 320; _canvas.height = 240;

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      }

      function capturarFrame() {
        _canvas.getContext('2d').drawImage(_video, 0, 0, _canvas.width, _canvas.height);
        return _canvas.toDataURL('image/jpeg', 0.8);
      }

      function atualizarLed(ligado, info) {
        _led.style.background = ligado ? '#ff4136' : '#3a1010';
        _led.style.boxShadow = ligado ? '0 0 28px 8px rgba(255,65,54,.65)' : 'none';
        _txt.textContent = ligado ? 'LIGADO' : 'DESLIGADO';
        _info.textContent = info;
      }

      function pararVideo() {
        if (_stream) _stream.getTracks().forEach(t => t.stop());
        if (_video) _video.remove();
      }
    '''))

    eval_js("iniciarVideo()")
    resetar_led()
    fim = time.time() + segundos

    try:
        while time.time() < fim:
            dados = eval_js("capturarFrame()")
            if not dados:
                break
            imagem = Image.open(io.BytesIO(base64.b64decode(dados.split(",", 1)[1])))

            probs = prever(imagem)
            ligado, vencedora = decidir_led(probs)
            info = f"{CLASSES[vencedora]} - {probs[vencedora]:.0%}"

            eval_js(f"atualizarLed({json.dumps(ligado)}, {json.dumps(info)})")
    except KeyboardInterrupt:
        print("Encerrado pelo usuario.")
    finally:
        eval_js("pararVideo()")
        print("Camera desligada.")


demo_ao_vivo(segundos=60)

---

### Se algo der errado na hora da apresentacao

| Problema | O que fazer |
|---|---|
| Erro ao carregar o `.h5` | A celula 3 ja tenta 3 estrategias e imprime os erros. Se todas falharem, rode `!pip install -q tf_keras` e execute a celula de novo. |
| `FileNotFoundError: keras_model.h5` | O upload nao terminou ou o arquivo tem outro nome. Confira a lista impressa na celula 2. |
| A camera nao aparece | Permita o acesso a camera no cadeado da barra de enderecos e rode a celula de novo. Nao pode haver duas celulas usando a camera ao mesmo tempo. |
| O LED nao acende | Baixe o `LIMIAR` (ex.: `0.5`) e confirme se `CLASSE_LIGA` aponta para a classe certa. |
| A demo ao vivo esta lenta | Normal: cada quadro vai do navegador ate o Python. Se travar, use a demo por foto. |